In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CUSTOM_MONTHS = 3


OFFTAKE_FILE = Path("C:/Users/pawarux2/OneDrive - Abbott/Documents/FFC/FFC Test/output/offtake_step2_final.csv")
DB_LEVEL_FILE = Path("C:/Users/pawarux2/OneDrive - Abbott/Documents/FFC/FFC Test/output/DB_Level_Apollo_Keimed_2026_local.csv")
PIN_TERR_FILE = Path("C:/Users/pawarux2/OneDrive - Abbott/Documents/FFC/FFC Test/input/20260402_Mar Pincode_Universe_AIL Mapped.csv")

OUT_CONTRI = Path("C:/Users/pawarux2/OneDrive - Abbott/Documents/FFC/FFC Test/output/FF_Contribution_All_Channels_local.csv")

custom_months = 3

for p in [OFFTAKE_FILE, DB_LEVEL_FILE, PIN_TERR_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")

In [2]:
# -------- LOAD STEP-2 OFFTAKE (SAFE) --------
offtake = pd.read_csv(OFFTAKE_FILE)

offtake = offtake[
    [
        "month",
        "name of customer",
        "Division Name",
        "Affiliate",
        "pincode",
        "channel_sales",
    ]
].copy()

offtake["month"] = pd.to_datetime(offtake["month"], errors="coerce")
offtake["pincode"] = offtake["pincode"].astype("string")


# -------- LOAD STEP-3 DB LEVEL (SAFE) --------
db_level = pd.read_csv(DB_LEVEL_FILE)

db_level = db_level[
    [
        "month",
        "name of customer",
        "Division Name",
        "Affiliate",
        "pincode",
        "channel_sales",
    ]
].copy()

db_level["month"] = pd.to_datetime(db_level["month"], errors="coerce")
db_level["pincode"] = db_level["pincode"].astype("string")


# -------- LOAD PINCODE → TERRITORY --------
dvl_from_ff = pd.read_csv(
    PIN_TERR_FILE,
    usecols=["pincode", "Division Name", "Affiliate", "Final Mapped Territory"],
    dtype={"pincode": "string"},
)

dvl_from_ff.rename(
    columns={"Final Mapped Territory": "Territory Code"}, inplace=True
)

offtake.shape, db_level.shape, dvl_from_ff.shape

C:\Users\pawarux2\AppData\Local\Temp\ipykernel_4504\2067622607.py:2: DtypeWarning: Columns (0: State, 1: District, 2: SKU No, 3: Brand Name, 4: Division Name) have mixed types. Specify dtype option on import or set low_memory=False.
  offtake = pd.read_csv(OFFTAKE_FILE)


((203747, 6), (1081536, 6), (160824, 4))

In [3]:
def normalize_customer(name):
    if pd.isna(name):
        return name
    n = name.upper().strip()
    if "APOLLO" in n:
        return "APOLLO"
    if "KEIMED" in n:
        return "KEIMEDGT"
    if "FLIPKART" in n:
        return "FLIPKART HEALTH PLUS"
    if "NETMEDS" in n:
        return "NETMEDS"
    return n

for df in [offtake, db_level]:
    df["name of customer"] = df["name of customer"].apply(normalize_customer)
    df["Division Name"] = df["Division Name"].str.upper().str.strip()
    df["Affiliate"] = df["Affiliate"].str.upper().str.strip()

In [4]:
res = pd.concat([offtake, db_level], ignore_index=True)

last_date = res["month"].max()

offtake_data_filtered = res[
    (res["month"] > last_date - pd.offsets.MonthBegin(CUSTOM_MONTHS))
    & (res["month"] <= last_date)
].copy()

offtake_data_filtered["channel_sales1"] = (
    offtake_data_filtered["channel_sales"] / CUSTOM_MONTHS
)

In [5]:
offtake_data_filtered["channel_sales"].sum()

np.float64(557167400.6674)

In [6]:
# Count how many territories per Division + pincode
dvl_grouped_bu_pin = (
    dvl_from_ff
    .groupby(["Division Name", "pincode"])
    .agg(count=("Territory Code", "nunique"))
    .reset_index()
)

dvl_grouped_bu_pin["count"] = dvl_grouped_bu_pin["count"].replace(0, 1)


In [7]:
offtake_grouped = pd.merge(
    offtake_data_filtered,
    dvl_grouped_bu_pin,
    how="left",
    on=["Division Name", "pincode"],
)

offtake_grouped["count"] = offtake_grouped["count"].fillna(1)

offtake_grouped["app_chan_sales"] = (
    offtake_grouped["channel_sales1"] / offtake_grouped["count"]
)

In [8]:
offtake_grouped_new = pd.merge(
    offtake_grouped,
    dvl_from_ff[["pincode", "Division Name", "Affiliate", "Territory Code"]],
    how="left",
    on=["pincode", "Division Name", "Affiliate"],
)

# ✅ IMPORTANT:
# Do NOT globally drop unmapped yet


In [9]:
cols_to_keep = [
    "name of customer",
    "Division Name",
    "Territory Code",
    "pincode",
    "channel_sales",
    "app_chan_sales",
]

offtake_grouped_req = offtake_grouped_new[cols_to_keep].copy()

In [10]:
offtake_grouped_req["Territory Code"].notna().sum()

np.int64(52289)

In [11]:
ap_keimed = ["APOLLO", "KEIMEDGT"]
other_accounts = ["1MG", "MEDPLUS", "PHARMEASY", "FLIPKART HEALTH PLUS", "UDAAN", "WELLNESS", "NETMEDS"]

offtake_ap = offtake_grouped_req[
    offtake_grouped_req["name of customer"].isin(ap_keimed)
].copy()

offtake_other = offtake_grouped_req[
    offtake_grouped_req["name of customer"].isin(other_accounts)
].copy()

In [12]:
offtake_ap = offtake_ap[offtake_ap["Territory Code"].notna()]
offtake_other = offtake_other[offtake_other["Territory Code"].notna()]


In [13]:
# terr_ap = (
#     offtake_ap
#     .groupby(["name of customer", "Division Name", "Territory Code"])
#     [["app_chan_sales"]]          # ✅ USE app_chan_sales ONLY
#     .sum()
#     .reset_index()
# )

# pivot_ap = terr_ap.pivot(
#     index=["Division Name", "Territory Code"],
#     columns="name of customer",
#     values="app_chan_sales",
# ).fillna(0).reset_index()

# for c in ["APOLLO", "KEIMEDGT"]:
#     if c not in pivot_ap.columns:
#         pivot_ap[c] = 0

# pivot_ap["final_channel_sales"] = pivot_ap["APOLLO"] + pivot_ap["KEIMEDGT"]
# pivot_ap["total_sales"] = (
#     pivot_ap
#     .groupby("Division Name")["final_channel_sales"]
#     .transform("sum")
# )

# pivot_ap["contri"] = np.where(
#     pivot_ap["total_sales"] == 0,
#     0,
#     pivot_ap["final_channel_sales"] / pivot_ap["total_sales"]
# )

# pivot_ap["name of customer"] = "KEIMED"

# terr_ap_final = pivot_ap[
#     ["name of customer", "Division Name", "Territory Code",
#      "final_channel_sales", "total_sales", "contri"]
# ]
terr_ap = (
    offtake_ap
    .assign(app_chan_sales=lambda x: pd.to_numeric(x["app_chan_sales"], errors="coerce").fillna(0))
    .groupby(["name of customer", "Division Name", "Territory Code"], dropna=False)[["app_chan_sales"]]
    .sum()
    .reset_index()
)

pivot_ap = terr_ap.pivot(
    index=["Division Name", "Territory Code"],
    columns="name of customer",
    values="app_chan_sales",
).fillna(0).reset_index()

for c in ["APOLLO", "KEIMEDGT"]:
    if c not in pivot_ap.columns:
        pivot_ap[c] = 0

pivot_ap["final_channel_sales"] = pivot_ap["APOLLO"] + pivot_ap["KEIMEDGT"]

pivot_ap["total_sales"] = (
    pivot_ap
    .groupby("Division Name")["final_channel_sales"]
    .transform("sum")
)

pivot_ap["contri"] = np.where(
    pivot_ap["total_sales"] == 0,
    0,
    pivot_ap["final_channel_sales"] / pivot_ap["total_sales"]
)

pivot_ap["name of customer"] = "KEIMED"

terr_ap_final = pivot_ap[
    [
        "name of customer",
        "Division Name",
        "Territory Code",
        "final_channel_sales",
        "total_sales",
        "contri",
    ]
].copy()

In [14]:
terr_other = (
    offtake_other
    .assign(app_chan_sales=lambda x: pd.to_numeric(x["app_chan_sales"], errors="coerce").fillna(0))
    .groupby(["name of customer", "Division Name", "Territory Code"], dropna=False)[["app_chan_sales"]]
    .sum()
    .reset_index()
)

terr_other["total_sales"] = (
    terr_other
    .groupby(["name of customer", "Division Name"])["app_chan_sales"]
    .transform("sum")
)

terr_other["contri"] = np.where(
    terr_other["total_sales"] == 0,
    0,
    terr_other["app_chan_sales"] / terr_other["total_sales"]
)

terr_other.rename(
    columns={"app_chan_sales": "final_channel_sales"},
    inplace=True
)

In [15]:
offtake_grouped_req["name of customer"].value_counts().head(20)

name of customer
APOLLO                  888534
KIEMEDGT                212213
TRUEMEDS                 59410
WELLNESS                 40004
NETMEDS                  31015
MEDPLUS                  18392
TATA 1MG                 11800
FLIPKART HEALTH PLUS      6441
PHARMEASY                 5679
ASTER                     2996
FRANK ROSS                2046
QCOMMERCE                 1361
NOBLE                     1301
ZENO HEALTH               1046
ASWAS                      847
THULASI                    846
MS PHARMACY                525
MUTHU PHARMACY             271
PULSE PHARMACY             243
GUARDIAN                   187
Name: count, dtype: int64

In [16]:
offtake_ap = offtake_ap[offtake_ap["Territory Code"].notna()]
offtake_other = offtake_other[offtake_other["Territory Code"].notna()]

In [17]:
# ============================================================
# STEP 5 DEBUG AUDIT — FIND WHERE SALES BECOME ZERO
# ============================================================

def audit_df(label, df):
    print("\n" + "=" * 80)
    print(label)
    print("shape:", df.shape)

    for col in ["channel_sales", "channel_sales1", "app_chan_sales"]:
        if col in df.columns:
            print(f"{col} sum:", pd.to_numeric(df[col], errors="coerce").fillna(0).sum())
            print(f"{col} non-zero rows:", (pd.to_numeric(df[col], errors="coerce").fillna(0) != 0).sum())

    if "name of customer" in df.columns:
        print("\nTop customers:")
        print(df["name of customer"].value_counts().head(15))

    if "Territory Code" in df.columns:
        print("\nMapped territory rows:", df["Territory Code"].notna().sum())

audit_df("1. OFFTAKE RAW", offtake)
audit_df("2. DB LEVEL RAW", db_level)
audit_df("3. COMBINED RES", res)
audit_df("4. LAST 3 MONTH FILTERED", offtake_data_filtered)
audit_df("5. AFTER TERRITORY MERGE - ALL ROWS", offtake_grouped_req)
audit_df("6. APOLLO + KEIMED AFTER MAPPED FILTER", offtake_ap)
audit_df("7. OTHER ACCOUNTS AFTER MAPPED FILTER", offtake_other)

print("\n" + "=" * 80)
print("Customer-wise channel_sales after mapped filter")
print(
    pd.concat([offtake_ap, offtake_other], ignore_index=True)
    .assign(channel_sales=lambda x: pd.to_numeric(x["channel_sales"], errors="coerce").fillna(0))
    .groupby("name of customer")["channel_sales"]
    .sum()
    .sort_values(ascending=False)
)


1. OFFTAKE RAW
shape: (203747, 6)
channel_sales sum: 67894900.27
channel_sales non-zero rows: 124074

Top customers:
name of customer
TRUEMEDS                59410
WELLNESS                40004
NETMEDS                 31015
APOLLO                  19211
MEDPLUS                 18392
TATA 1MG                11800
FLIPKART HEALTH PLUS     6441
PHARMEASY                5679
ASTER                    2996
FRANK ROSS               2046
QCOMMERCE                1361
NOBLE                    1301
ZENO HEALTH              1046
ASWAS                     847
THULASI                   846
Name: count, dtype: int64

2. DB LEVEL RAW
shape: (1081536, 6)
channel_sales sum: 489272500.3974
channel_sales non-zero rows: 1080095

Top customers:
name of customer
APOLLO      869323
KIEMEDGT    212213
Name: count, dtype: int64

3. COMBINED RES
shape: (1285283, 6)
channel_sales sum: 557167400.6674
channel_sales non-zero rows: 1204169

Top customers:
name of customer
APOLLO                  888534
KIEMEDGT    

In [18]:
final_output = pd.concat([terr_other, terr_ap_final], ignore_index=True)

print("Final output shape:", final_output.shape)
print("Final channel sales sum:", final_output["final_channel_sales"].sum())
print("Total sales sum:", final_output["total_sales"].sum())
print("Contribution max:", final_output["contri"].max())

if final_output["final_channel_sales"].sum() == 0:
    raise ValueError(
        "Step 5 output has real territory rows but zero sales. "
        "This means mapped + allowed-customer rows have channel_sales = 0. "
        "Fix Step 2 / Step 3 channel_sales before rerunning Step 5."
    )

final_output.to_csv(OUT_CONTRI, index=False)
final_output.shape

Final output shape: (1344, 6)
Final channel sales sum: 6611345.559999999
Total sales sum: 310479190.42
Contribution max: 0.4720670750293834


(1344, 6)

In [19]:
final_output = pd.concat([terr_other, terr_ap_final], ignore_index=True)

final_output.to_csv(OUT_CONTRI, index=False)

final_output.shape

(1344, 6)

In [20]:
final_output["contri"].describe()

count    1344.000000
mean        0.011905
std         0.028030
min         0.000051
25%         0.001769
50%         0.004170
75%         0.009805
max         0.472067
Name: contri, dtype: float64

In [21]:
final_output.groupby("Division Name")["contri"].sum().head()

Division Name
GENNEXT        2.0
GI MAXIMA      2.0
GI OPTIMA      2.0
GI PRIMA       2.0
GI PROSPERA    2.0
Name: contri, dtype: float64

In [22]:
final_output["Territory Code"].value_counts().head()

Territory Code
IT007096    2
IT007097    2
IT007112    2
IT007117    2
IT007119    2
Name: count, dtype: int64